# DMOSpeech 2: Early Handover Development Experiment (GPU)
## Empirical Evaluation of Expanded Candidate Steps {2, 3, 4, 5, 6, 8} on 100 Development Speakers

**Author:** Rhenel Jhon Sajol  
**Specification:** `VERDICT_COLAB_EARLY_HANDOVER_DEVELOPMENT_TEST.md`  
**Evaluation Scope:** 100 Development Speakers (200 Cases: Clean & Noisy15)  
**Candidate Grid:** {2, 3, 4, 5, 6, 8}  
**Status:** **Development-Only Exploratory Analysis**  

### Purpose:
Determine whether handover before index 4 (indices 2 and 3) provides useful quality–computation headroom, and whether an adaptive rule (or Candidates A, B, C) can reliably identify safe early exits without compromising speech fidelity.

### Step 1: Verify CUDA GPU Allocation
Ensure an NVIDIA GPU (preferably Tesla T4) is allocated.

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")

### Step 2: Clone Thesis Repository & Setup Python Environment

In [ ]:
import os
if not os.path.exists('Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git

%cd Shonget-DMOSpeech2
!git pull

# Install required dependencies matching thesis environment
!pip install -q torchaudio transformers soundfile jiwer librosa scipy pandas pyarrow speechbrain pytest zhon

### Step 3: Fast Download Model Checkpoints from HuggingFace

In [ ]:
!mkdir -p ckpts
# Download DMOSpeech 2 Teacher/Student and Duration Predictor
!wget -c -q --show-progress https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c -q --show-progress https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

!ls -lh ckpts/

### Step 4: Download Seed-TTS Evaluation Dataset (Audio Prompts)

In [ ]:
import os
test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'
if not os.path.exists(test_file):
    print("Downloading Seed-TTS English evaluation dataset...")
    !wget -c -q --show-progress https://huggingface.co/datasets/yl4579/seed-tts-eval/resolve/main/seed-tts-eval.tar.gz
    !mkdir -p data
    !tar -xzf seed-tts-eval.tar.gz -C data/
    !rm seed-tts-eval.tar.gz

print("Seed-TTS prompt audio files verified on disk.")

### Step 5: Validate Development Manifest & Check Existing GPU Records
Verifies 100 distinct development speakers and checks anchor alignment with `gpu_grid_all1000.csv`.

In [ ]:
import json
import pandas as pd

manifest_path = 'Handover Scheduling/data/splits/development_manifest.json'
with open(manifest_path, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

print(f"Development Manifest Total Cases: {len(manifest)}")
spks = set(x['speaker_id'] for x in manifest)
print(f"Unique Speakers: {len(spks)}")

gpu_grid_path = 'Handover Scheduling/colab/results/gpu_grid_all1000.csv'
if os.path.exists(gpu_grid_path):
    df_existing = pd.read_csv(gpu_grid_path)
    dev_utts = set(x['utt_id'] for x in manifest)
    dev_existing = df_existing[df_existing['utt_id'].isin(dev_utts)]
    print(f"Reusable Development Records in GPU Grid: {len(dev_existing)} rows (Steps 4, 5, 6, 8)")
else:
    print("Existing GPU grid not found. All candidate steps {2, 3, 4, 5, 6, 8} will be generated.")

### Step 6: Verify Expanded Sampler Semantics (Anchor Check)
Verifies that Step 2 (9 NFE) and Step 3 (11 NFE) execute deterministically alongside existing steps.

In [ ]:
import sys
sys.path.insert(0, 'Handover Scheduling')
from core.sampler import STEP_METADATA, SWAY_TEACHER_GRID

print("=== Expanded Sway Timing & NFE Schedule ===")
for s in [2, 3, 4, 5, 6, 8]:
    m = STEP_METADATA[s]
    print(f"Step {s}: Sway Time = {m['actual_teacher_time']:.4f} | Teacher NFE = {m['teacher_nfe']} | Total NFE = {m['nfe']}")

### Step 7: Execute Expanded Development Grid Generation & Evaluation
Runs the complete pipeline on the 100 development speakers across steps {2, 3, 4, 5, 6, 8}.
Saves checkpointed evaluations to `Handover Scheduling/colab/results/early_handover_dev_gpu/development_expanded_grid.csv`.

In [ ]:
!python "Handover Scheduling/colab/run_early_handover_dev_gpu.py" \
  --manifest "Handover Scheduling/data/splits/development_manifest.json" \
  --reusable_grid "Handover Scheduling/colab/results/gpu_grid_all1000.csv" \
  --output_dir "Handover Scheduling/colab/results/early_handover_dev_gpu" \
  --device cuda

### Step 8: Early Signal Diagnostics (Discrimination at Steps 2 and 3)
Examines whether tangent drift ($d_2, d_3$), drift trend ($q_A$), local stability ($q_B$), or conditioning disagreement ($q_C$) identify acoustic failures.

In [ ]:
import pandas as pd
diag_path = 'Handover Scheduling/colab/results/early_handover_dev_gpu/early_signal_diagnostics.csv'
if os.path.exists(diag_path):
    df_diag = pd.read_csv(diag_path)
    display(df_diag)
else:
    print("Run Step 7 to generate early signal diagnostics.")

### Step 9: 5-Fold Cross-Validation of Candidate Formulas (A, B, C)
Evaluates generalization risk across 5 speaker-disjoint folds.

In [ ]:
fold_path = 'Handover Scheduling/colab/results/early_handover_dev_gpu/candidate_formula_fold_results.csv'
if os.path.exists(fold_path):
    df_folds = pd.read_csv(fold_path)
    summary = df_folds.groupby('candidate_formula')[['val_robust_risk_%', 'val_mean_nfe', 'val_clean_failures']].mean()
    display(summary)
else:
    print("Run Step 7 to generate 5-fold cross-validation results.")

### Step 10: Fair Baseline & Oracle Headroom Comparison
Compares Fixed Step 8, SNR-only 4/8, SNR-only 2/8, SNR-only 3/8, Original SNR+drift, and Expanded Oracle.

In [ ]:
pol_path = 'Handover Scheduling/colab/results/early_handover_dev_gpu/expanded_policy_comparison.csv'
if os.path.exists(pol_path):
    df_pol = pd.read_csv(pol_path)
    display(df_pol[['policy', 'mean_nfe', 'nfe_reduction_vs_step8_%', 'robust_failure_rate_%', 'clean_failure_rate_%', 'clean_mean_wer', 'clean_mean_sim']])
else:
    print("Run Step 7 to generate policy comparison table.")

### Step 11: Download All Generated Artifacts & Markdown Report
Zips the `early_handover_dev_gpu/` folder containing all CSVs and `EARLY_HANDOVER_DEVELOPMENT_REPORT.md`.

In [ ]:
!tar -czf early_handover_dev_gpu_artifacts.tar.gz -C "Handover Scheduling/colab/results" early_handover_dev_gpu

try:
    from google.colab import files
    files.download('early_handover_dev_gpu_artifacts.tar.gz')
    print("Download initiated.")
except Exception as e:
    print(f"File saved at early_handover_dev_gpu_artifacts.tar.gz: {e}")